<a href="https://colab.research.google.com/github/leon72305-source/DSAI_FINAL/blob/main/DSandAIP2027_Drafts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## *Datasets Description*

| Column | Meaning |
| ----------- | ----------- |
| `df_chargers` | Dataset for Public Charging Infrastructure Statistics |
| `df_piv` | Dataset for Licensed Plug-In Vehicles (PIVs) Statistics|

## Environment Setup

The `-q` flag keeps the output quiet.

In [ ]:
!pip install -q odfpy

In [ ]:
from google.colab import drive

drive.mount("/content/drive/")

In [110]:
import pandas as pd
import numpy as np

from datetime import datetime

import re

import matplotlib.pyplot as plt
import seaborn as sns

## Loading the datasets

In [111]:
folder_path = "/content/drive/MyDrive/DataSet/"

df_piv = pd.read_csv(folder_path + "veh0142.csv", header=4)
df_chargers = pd.read_csv(folder_path + "evci9001.csv", header=2)

print("df_piv shape:", df_piv.shape)
print("df_chargers shape:", df_chargers.shape)

display(df_piv.head(3))
display(df_chargers.head(3))

df_piv shape: (26156, 74)
df_chargers shape: (470, 30)


,Units,BodyType,Fuel,Keepership,ONS Sort,ONS Code,ONS Geography,2026 Q2,2026 Q1,2025 Q4,...,2012 Q1,2011 Q4,2011 Q3,2011 Q2,2011 Q1,2010 Q4,2010 Q3,2010 Q2,2010 Q1,2009 Q4
0,Number,Buses and coaches,Battery electric,Company,1,K02000001,United Kingdom,"8,817","8,235","7,669",...,[x],[x],[x],[x],[x],[x],[x],[x],[x],[x]
1,Number,Buses and coaches,Battery electric,Company,2,K03000001,Great Britain,"8,549","7,988","7,425",...,54,58,57,59,58,49,45,41,40,38
2,Number,Buses and coaches,Battery electric,Company,3,E92000001,England,"7,333","6,804","6,276",...,51,55,54,57,55,46,43,39,38,36


,Local authority / region code,Local authority / region name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,Jan-21,Apr-21,Jul-21,...,Apr-24,Jul-24,Oct-24 [Note 14],Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26,Jul-26
0,K02000001,United Kingdom,"15,116","16,505","17,947","18,265","19,487","20,775","22,790","24,374",...,"59,670","64,632","70,042","73,334","76,507","82,002","86,021","87,796","92,141","94,595"
1,K03000001,Great Britain,"14,821","16,210","17,642","17,953","19,169","20,455","22,463","24,044",...,"59,125","64,014","69,403","72,654","75,835","81,318","85,283","87,069","91,414","93,864"
2,E92000001,England,"12,549","13,719","14,979","15,395","16,456","17,459","19,261","20,563",...,"51,506","55,631","60,364","63,389","65,877","70,672","74,115","75,818","79,198","81,231"


## Data types and data quality audit

In [116]:
print("df_piv columns:", df_piv.columns[:7].tolist())
print("df_chargers columns:", df_chargers.columns[:5].tolist())

# 2. Standardize column names (Update the keys on the left to match your actual columns)
df_piv = df_piv.rename(columns={
    'ONS Code': 'Local authority / region code',          # Replace with whatever df_piv currently calls it
    'ONS Geography': 'Local authority / region name'
})

df_piv = df_piv.drop(columns=['Units', 'Unit'], errors='ignore')
df_chargers = df_chargers.drop(columns=['Units', 'Unit'], errors='ignore')

print("\nColumns successfully harmonized!")

df_piv columns: ['Units', 'BodyType', 'Fuel', 'Keepership', 'ONS Sort', 'Local authority / region code', 'Local authority / region name']
df_chargers columns: ['Local authority / region code', 'Local authority / region name', '2019 Q4', '2020 Q1', '2020 Q2']

Columns successfully harmonized!


In [117]:
quarter_map = {
    'jan': 1, 'feb': 1, 'mar': 1,
    'apr': 2, 'may': 2, 'jun': 2,
    'jul': 3, 'aug': 3, 'sep': 3,
    'oct': 4, 'nov': 4, 'dec': 4
}

# 2. Filter function for EV Adoption (df_piv)
def filter_veh(df):
    cols_to_keep = []
    for col in df.columns:
        col_str = str(col).strip()
        match = re.search(r'(20\d{2})\s*Q?([1-4])?', col_str)
        if match:
            year = int(match.group(1))
            q_match = match.group(2)
            quarter = int(q_match) if q_match else None

            if year > 2019:
                cols_to_keep.append(col)
            elif year == 2019:
                if quarter is None or quarter >= 4:
                    cols_to_keep.append(col)
        else:
            cols_to_keep.append(col)
    return df[cols_to_keep]

# 3. Filter and standardize function for Chargers (df_chargers)
def filter_chargers(df):
    rename_mapping = {}
    cols_to_drop = []
    for col in df.columns:
        col_str = str(col).strip().lower()
        match = re.search(r'([a-z]{3})[-/\s]?(\d{2,4})', col_str)
        if match:
            month_str = match.group(1)
            year_raw = match.group(2)

            if month_str in quarter_map:
                quarter = quarter_map[month_str]
                year = int(year_raw)
                if year < 100:
                    year += 2000

                # Rule: Keep up to April 2026 (2026 Q2)
                if (year < 2026) or (year == 2026 and quarter <= 2):
                    new_col_name = f"{year} Q{quarter}"
                    rename_mapping[col] = new_col_name
                else:
                    cols_to_drop.append(col)

    df_filtered = df.drop(columns=cols_to_drop, errors='ignore')
    df_filtered = df_filtered.rename(columns=rename_mapping)

    # Separate metadata to prevent alphabetical scrambling
    meta_cols = [c for c in df_filtered.columns if c not in rename_mapping.values()]
    time_cols = [c for c in df_filtered.columns if c in rename_mapping.values()]

    df_meta = df_filtered[meta_cols]
    df_times = df_filtered[time_cols].groupby(by=df_filtered[time_cols].columns, axis=1).last()

    return pd.concat([df_meta, df_times], axis=1)

# Run the initial cleaning step
df_piv = filter_veh(df_piv)
df_chargers = filter_chargers(df_chargers)

print(f"Cleaned df_piv shape: {df_piv.shape}")
print(f"Cleaned df_chargers shape: {df_chargers.shape}")

display(df_piv.head(3))
display(df_chargers.head(3))

Cleaned df_piv shape: (26156, 33)
Cleaned df_chargers shape: (470, 29)


/tmp/ipykernel_2580/2071171450.py:60: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_times = df_filtered[time_cols].groupby(by=df_filtered[time_cols].columns, axis=1).last()


,BodyType,Fuel,Keepership,ONS Sort,Local authority / region code,Local authority / region name,2026 Q2,2026 Q1,2025 Q4,2025 Q3,...,2022 Q1,2021 Q4,2021 Q3,2021 Q2,2021 Q1,2020 Q4,2020 Q3,2020 Q2,2020 Q1,2019 Q4
0,Buses and coaches,Battery electric,Company,1,K02000001,United Kingdom,"8,817","8,235","7,669","6,997",...,"1,519","1,241","1,025",933,871,786,715,661,630,482
1,Buses and coaches,Battery electric,Company,2,K03000001,Great Britain,"8,549","7,988","7,425","6,752",...,"1,481","1,237","1,023",933,871,786,715,661,630,482
2,Buses and coaches,Battery electric,Company,3,E92000001,England,"7,333","6,804","6,276","5,669",...,"1,171","1,044",929,871,812,734,670,634,599,452


,Local authority / region code,Local authority / region name,2019 Q4,2020 Q1,2020 Q2,2020 Q3,2020 Q4,2021 Q1,2021 Q2,2021 Q3,...,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2
0,K02000001,United Kingdom,"15,116","16,505","17,947","18,265","19,487","20,775","22,790","24,374",...,"53,677","59,670","64,632","70,042","73,334","76,507","82,002","86,021","87,796","92,141"
1,K03000001,Great Britain,"14,821","16,210","17,642","17,953","19,169","20,455","22,463","24,044",...,"53,212","59,125","64,014","69,403","72,654","75,835","81,318","85,283","87,069","91,414"
2,E92000001,England,"12,549","13,719","14,979","15,395","16,456","17,459","19,261","20,563",...,"46,374","51,506","55,631","60,364","63,389","65,877","70,672","74,115","75,818","79,198"


## Data reshaping

In [118]:
def melt_dataframe(df, value_name):
    # Drop any duplicate columns if code/name fields were repeated
    df = df.loc[:, ~df.columns.duplicated()]

    # Identify metadata columns (anything that is NOT a quarterly time column like "YYYY Q#")
    id_vars = [col for col in df.columns if not re.match(r'^\d{4}\s+Q[1-4]$', str(col).strip())]
    value_vars = [col for col in df.columns if col not in id_vars]

    # Melt from wide to long format
    df_long = pd.melt(
        df,
        id_vars=id_vars,
        value_vars=value_vars,
        var_name='quarter',
        value_name=value_name
    )

    # Clean up value column: remove commas and convert to numeric safely
    df_long[value_name] = pd.to_numeric(
        df_long[value_name].astype(str).str.replace(',', ''),
        errors='coerce'
    )

    return df_long

# Apply melting to both datasets
df_piv_long = melt_dataframe(df_piv, 'total_evs')
df_chargers_long = melt_dataframe(df_chargers, 'total_chargers')

print(f"df_piv_long shape: {df_piv_long.shape}")
print(f"df_chargers_long shape: {df_chargers_long.shape}")

display(df_piv_long.head(25))
display(df_chargers_long.head(3))

df_piv_long shape: (706212, 8)
df_chargers_long shape: (12690, 4)


,BodyType,Fuel,Keepership,ONS Sort,Local authority / region code,Local authority / region name,quarter,total_evs
0,Buses and coaches,Battery electric,Company,1,K02000001,United Kingdom,2026 Q2,8817
1,Buses and coaches,Battery electric,Company,2,K03000001,Great Britain,2026 Q2,8549
2,Buses and coaches,Battery electric,Company,3,E92000001,England,2026 Q2,7333
3,Buses and coaches,Battery electric,Company,4,E12000001,North East,2026 Q2,211
4,Buses and coaches,Battery electric,Company,5,E06000047,County Durham,2026 Q2,1
5,Buses and coaches,Battery electric,Company,6,E06000003,Redcar and Cleveland,2026 Q2,1
6,Buses and coaches,Battery electric,Company,7,E06000004,Stockton-on-Tees,2026 Q2,0
7,Buses and coaches,Battery electric,Company,8,E11000007,Tyne and Wear (Met County),2026 Q2,208
8,Buses and coaches,Battery electric,Company,9,E08000037,Gateshead,2026 Q2,48
9,Buses and coaches,Battery electric,Company,10,E08000021,Newcastle upon Tyne,2026 Q2,2


,Local authority / region code,Local authority / region name,quarter,total_chargers
0,K02000001,United Kingdom,2019 Q4,15116.0
1,K03000001,Great Britain,2019 Q4,14821.0
2,E92000001,England,2019 Q4,12549.0
